In [ ]:
%pip install optuna fastparquet --quiet

In [ ]:
import os
import sys
import glob
import shutil

SRC_ROOT = '/kaggle/input/datasets/mmoselli/mouse-dynamics-ic'
DST_ROOT = '/kaggle/working/mouse-dynamics'

os.makedirs(DST_ROOT, exist_ok=True)

shutil.copytree(
    f'{SRC_ROOT}/src',
    f'{DST_ROOT}/src',
    dirs_exist_ok=True
)

for src_db in glob.glob(f'{SRC_ROOT}/best-parameters/**/*.db', recursive=True):
    relative = os.path.relpath(src_db, f'{SRC_ROOT}/best-parameters')
    dst_db   = os.path.join(DST_ROOT, 'best-parameters', relative)
    os.makedirs(os.path.dirname(dst_db), exist_ok=True)
    if not os.path.exists(dst_db):
        shutil.copy2(src_db, dst_db)

print(f"src/     copiado : {os.path.exists(f'{DST_ROOT}/src')}")
print(f"db files copiados: {len(glob.glob(f'{DST_ROOT}/best-parameters/**/*.db', recursive=True))}")

In [ ]:
import os

src = '/kaggle/input/datasets/mmoselli/mouse-dynamics-bogazici-split/split'
dst = '/kaggle/working/datasets/split'

os.makedirs('/kaggle/working/datasets', exist_ok=True)

if os.path.islink(dst) or os.path.exists(dst):
    os.remove(dst)

os.symlink(src, dst)

print(f"aponta para: {os.readlink(dst)}")
print(f"conteúdo: {os.listdir(dst)[:5]}")

In [ ]:
os.chdir('/kaggle/working/mouse-dynamics')

if '/kaggle/working/mouse-dynamics' not in sys.path:
    sys.path.insert(0, '/kaggle/working/mouse-dynamics')

from src.classifiers    import EnumClassifiers
from src.dataset_loaders import EnumDatasets
from src.preprocessors  import EnumPreprocessors
from src.splitters      import EnumSplitters
from src.orchestrator   import Orchestrator

print("Imports OK")
print(f"  EnumClassifiers : {[e.value for e in EnumClassifiers]}")
print(f"  EnumDatasets    : {[e.value for e in EnumDatasets]}")
print(f"  EnumPreprocessors: {[e.value for e in EnumPreprocessors]}")
print(f"  EnumSplitters   : {[e.value for e in EnumSplitters]}")

In [ ]:
import os

for dataset in ['balabit', 'bogazici', 'minecraft']:
    path = f'/kaggle/working/mouse-dynamics/best-parameters/{dataset}'
    os.makedirs(path, exist_ok=True)
    print(f"criado: {path}")

In [ ]:
from pathlib import Path
import os

p = Path("../datasets/split")
print(f"existe: {p.exists()}")
print(f"é symlink: {p.is_symlink()}")
print(f"aponta para: {os.readlink(p)}")
print(f"conteúdo: {list(p.iterdir())}")

In [ ]:
import gc

from src.classifiers import load_classifier
from src.utils.experiment_logger import ExperimentLogger
from src.dto.extraction_data import ExtractionData
from src.dto.user_data_dto import UserDataDto
from pandas import read_parquet

ALL_CLASSIFIERS  = [EnumClassifiers.KNN]

window_size = 50

print("=" * 80)
print(f"window_size={window_size}")
print("=" * 80)

split_path = Path("../datasets/split")
user_dirs = sorted(split_path.glob("user*"))

for classifier in ALL_CLASSIFIERS:
    with ExperimentLogger(
        classifier_name=classifier.value,
        dataset_name=EnumDatasets.BOGAZICI.value,
        preprocessor_name=EnumPreprocessors.KHAN.value,
        splitter_name=EnumSplitters.HALF.value,
        preprocessor_window_size=window_size,
        is_debug=False,
    ) as experiment_logger:
        loaded_classifier = load_classifier(classifier, False)
        loaded_classifier.set_experiment_logger(experiment_logger)

        for user_dir in user_dirs:
            user_id = user_dir.stem.replace("user", "")
            
            user_dto = UserDataDto(user_id)
            user_dto.training_sessions = {"_merged": read_parquet(user_dir / "training" / str(window_size)  / "_merged.parquet")}
            user_dto.testing_sessions  = {"_merged": read_parquet(user_dir / "testing" / str(window_size)   / "_merged.parquet")}

            extraction_data = ExtractionData()
            extraction_data.add_user(user_dto)

            loaded_classifier.fit(extraction_data)

            del user_dto, extraction_data
            gc.collect()

    del loaded_classifier
    gc.collect()

print(f"Concluído: window_size={window_size}\n")

print("=" * 80)
print("TODOS OS EXPERIMENTOS CONCLUÍDOS")
print("=" * 80)